# PII demo walkthrough

Run this notebook **as yourself** (a normal user, not the pipeline service principal). It shows what a human sees in each zone:

| Step | What you show | Zone |
| --- | --- | --- |
| 1 | The PII contract drives everything | `pii_governance` |
| 2 | Landing data is locked down by ABAC (secure by default) | `pii_raw`, `pii_vault` |
| 3 | The verification gate decides what may be published | `pii_governance.batch_control` |
| 4 | Analysts get tokens and ciphertext only, and can still join and count | `pii_analytics` |
| 5 | Only the service principal can re-identify or forget a person | `pii_vault` |

Prerequisite: `pii_setup_job` and `pii_main_job` have run (and optionally `pii_scenarios_job` for batch 2).

In [0]:
dbutils.widgets.text("catalog", "serverless_stable_yit814_catalog")
catalog = dbutils.widgets.get("catalog")
spark.sql(f"USE CATALOG {catalog}")
print(f"Running as: {spark.sql('SELECT current_user()').first()[0]}  |  catalog: {catalog}")

## Step 1: the PII contract (preventive control)

Every source column is declared here, with how it is normalized and protected. Columns that are **not** in the contract are routed to `_rescued_data` and quarantined by the raw pipeline. `class_tag` is the system governed tag applied to the raw column.

In [0]:
%sql
SELECT table_name, column_name, pii_type, class_tag, method, normalize_rule, key_or_vault
FROM pii_governance.pii_contract
WHERE active
ORDER BY table_name, method DESC, column_name

## Step 2: the restricted zone is secure by default

The Lakeflow Connect landing tables hold cleartext PII, and PII cannot be masked before ingestion. Two schema-level ABAC policies close that gap:

* a **column mask** on every column (`MATCH COLUMNS TRUE`, type-aware `system.data_classification.mask_value`)
* a **row filter** that returns no rows

Both apply `TO account users EXCEPT <pipeline service principal>` and have no `WHEN` clause, so every table that lands in the schema later is protected the moment it is created. You have `SELECT` on `pii_raw`, so ABAC (not a missing grant) is what hides the data.

In [0]:
%sql
-- Cleartext PII is in this table, but you see nothing
SELECT count(*) AS rows_visible_to_me FROM pii_raw.lfc_customers

In [0]:
%sql
SHOW EFFECTIVE POLICIES ON TABLE pii_raw.lfc_customers

In [0]:
%sql
DESCRIBE POLICY pii_raw_mask_all_columns ON SCHEMA pii_raw

In [0]:
%sql
-- Applied from the contract with system governed class.* tags (the same tags Data Classification uses)
SELECT table_name, column_name, tag_name
FROM information_schema.column_tags
WHERE schema_name = 'pii_raw'
ORDER BY table_name, column_name

In [0]:
%sql
-- Token -> value mapping. Same lockdown policies, and no grants for anyone but the service principal.
SELECT count(*) AS vault_rows_visible_to_me FROM pii_vault.tokmap_email

## Step 3: cleartext checks, quarantine and the verification gate

Format checks (email, E.164 phone, SSN) run on the **real, normalized** values inside the restricted zone, because once a value is a token it always looks valid. Failed rows go to `*_quarantine` (still in `pii_raw`, so you cannot see them either).

The verify pipeline then writes one row per table and batch with counts only, no values. The publish pipeline reads **only `VERIFIED` batches**. If you ran the scenarios job, batch 2 of `transactions` is `FAILED` on `no_cleartext_leak`: someone typed an email address into the `merchant` field, a column the contract says to keep.

In [0]:
%sql
SELECT table_name, batch_id, status, failed_checks,
       landed_rows, valid_rows, quarantined_rows, leak_rows, missing_token_rows, vault_conflicts, orphan_rows
FROM pii_governance.batch_control
ORDER BY table_name, batch_id

## Step 4: the pseudonymized zone

* `email`, `full_name`: **Pattern B, token vault**. A random `uuid` per normalized value, so the same person always gets the same token.
* `ssn`, `phone`, `dob`: **Pattern A, AES-256-GCM** with keys stored as Unity Catalog secrets (`*_enc`), plus a keyed **HMAC** (`*_hmac`). The ciphertext uses a random IV and can't be joined; the HMAC can.
* `street_address`, `card_number`: **dropped** (data minimization).

In [0]:
%sql
SELECT customer_id, full_name_token, email_token, ssn_enc, ssn_hmac, phone_enc, city, segment, signup_date, pii_key_version, _batch_id
FROM pii_analytics.customers
ORDER BY customer_id
LIMIT 10

In [0]:
%sql
-- Distinct counts and joins work on tokens and HMACs, without anyone re-identifying.
-- Here every row is a different person, so all counts match. The difference: encrypting the same SSN twice
-- gives a different ssn_enc (random IV), but always the same ssn_hmac, so only the HMAC can join across tables.
SELECT count(*)                    AS customers,
       count(DISTINCT email_token) AS distinct_people_by_email,
       count(DISTINCT ssn_hmac)    AS distinct_people_by_ssn,
       count(DISTINCT ssn_enc)     AS distinct_ciphertexts
FROM pii_analytics.customers

In [0]:
%sql
SELECT * FROM pii_analytics.customer_spend_by_segment ORDER BY segment, currency

In [0]:
%sql
-- Tags on raw tables do not carry over, so the published columns are tagged again (ungoverned tags)
SELECT column_name, tag_name, tag_value
FROM information_schema.column_tags
WHERE schema_name = 'pii_analytics' AND table_name = 'customers'
ORDER BY column_name, tag_name

## Step 5: re-identification and the right to be forgotten (service principal only)

* **Re-identify:** the `reidentify_check` task of `pii_main_job` (notebook `05_reidentify`) runs as the service principal. It decrypts with `pii_vault.pii_decrypt` and joins the vault, then checks that every published value round-trips to the original. It reports counts only.
* **Forget a person:** run as the service principal, delete their rows from the vault tables and the raw zone, then `VACUUM`. Their tokens in `pii_analytics` stay, but can no longer be reversed:

```sql
DELETE FROM pii_vault.tokmap_email WHERE token = '<email_token>';
DELETE FROM pii_vault.tokmap_name  WHERE token = '<full_name_token>';
```

* **Known limitation:** ABAC does not cover secrets or functions. Anyone with `ALL PRIVILEGES` on the catalog inherits `READ SECRET` on the keys. In production, put `pii_vault` in its own catalog that only the service principal can use.